# Define custom kernels (notebook)

- [Using a uv virtual environment](#uv-managed-virtual-environment)

## uv managed virtual environment

We can use uv to create a virtual environment for different versions of Python.

Let's create a new virtual environment in the folder named ".venv_py315" using uv.

**Requires a recent version of uv to be installed.**

In [ ]:
%uv venv .venv_py315 --python 3.15 --clear
%uv pip install --directory .venv_py315 --python 3.15 async-kernel

Now we can write a kernel spec that uses uv to start the kernel from the virtual environment.

In [ ]:
import pathlib

from async_kernel.kernelspec import write_kernel_spec

uv_path = pathlib.Path.cwd().joinpath(".venv_py315")
assert uv_path.exists()

write_kernel_spec(
    name="async_3.15",
    display_name="Python 3.15 (async)",
    env={"UV_PROJECT_ENVIRONMENT": str(uv_path)},
    executable=("uv", "run", "--no-sync", "python", "-m", "async_kernel"),
)

The kernel spec with the display name "Python 3.15 (async kernel)" has been added. You will need to refresh the list of kernels for it to be available.

## Embed the custom kernel in the kernelspec folder


A callable can be passed to `write_kernel_spec` as the argument `start_interface`, which is written to the kernel spec folder and used when starting the kernel.

Let's write a kernel that will echo (print) the code written to the cell. 


In [ ]:
import async_kernel.kernelspec


def start_interface(settings):
    from typing import override

    from async_kernel import Kernel
    from async_kernel.interface.zmq import ZMQKernelInterface

    class EchoKernel(Kernel):
        @override
        async def execute_request(self, job):
            print(job["msg"]["content"]["code"])
            return {"status": "ok", "execution_count": 0, "user_expressions": {}}

    # Subclassing the kernel will override it.

    # Start the kernel with a zmq interface.
    ZMQKernelInterface(settings).start()


# Write the kernel spec
async_kernel.kernelspec.write_kernel_spec(name="echo", start_interface=start_interface)